<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.9-long-running-async-agents/notebooks/exercises-9_9.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.9: Long-Running & Async Agents

*Module 9 · AI Agents & Autonomous Systems*

> Most agents finish in seconds. But some run for hours: processing 10,000 documents, crawling 500 websites, generating a 50-page report. These agents must checkpoint state , survive crashes, report progress, and resume where they left off. This lesson builds the infrastructure: durable state, async execution, Pub/Sub triggers, and Cloud Tasks for scaling.

`Checkpointing` · `Async` · `Pub/Sub` · `Cloud Tasks` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.9.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Durable Agent State — Checkpoint and Resume — `01_durable_agent.py`
2. Step 2 — Async Job API — Submit, Poll, Retrieve — `02_async_api.py`
3. Step 3 — Pub/Sub Triggers — Event-Driven Agent Execution — `03_pubsub_agent.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q fastapi pydantic


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Durable Agent State — Checkpoint and Resume

Save agent state after every step. Resume from any checkpoint on crash.

**`01_durable_agent.py`** · _Block 1 of 3_

DURABLE AGENT — Checkpoint after every step, resume on crash


In [ ]:
# DURABLE AGENT — Checkpoint after every step, resume on crash
import json, time, os, uuid
from pathlib import Path

class DurableAgent:
    """Agent that survives crashes via checkpointing."""
    def __init__(self, job_id=None, checkpoint_dir="./checkpoints"):
        self.job_id = job_id or str(uuid.uuid4())[:8]
        self.dir = Path(checkpoint_dir) / self.job_id
        self.dir.mkdir(parents=True, exist_ok=True)
        self.state = self._load_state()

    def _state_path(self): return self.dir / "state.json"

    def _load_state(self):
        if self._state_path().exists():
            with open(self._state_path()) as f: return json.load(f)
        return {"status":"pending", "step":0, "results":[], "errors":[], "started":time.time()}

    def _save_state(self):
        with open(self._state_path(), "w") as f: json.dump(self.state, f, indent=2)

    def checkpoint(self, step_result):
        """Save progress after each step."""
        self.state["step"] += 1
        self.state["results"].append(step_result)
        self.state["status"] = "running"
        self.state["last_checkpoint"] = time.time()
        self._save_state()

    def complete(self, final_result):
        self.state["status"] = "completed"
        self.state["final"] = final_result
        self.state["duration"] = time.time() - self.state["started"]
        self._save_state()

    def fail(self, error):
        self.state["status"] = "failed"
        self.state["errors"].append(str(error))
        self._save_state()

    def run(self, tasks):
        """Process tasks, resuming from last checkpoint."""
        start_from = self.state["step"]
        print(f"  Job {self.job_id}: resuming from step {start_from}/{len(tasks)}")

        for i in range(start_from, len(tasks)):
            try:
                result = f"Processed: {tasks[i]}"  # Replace with real work
                time.sleep(0.1)  # Simulate work
                self.checkpoint({"task":tasks[i], "result":result, "step":i+1})
                print(f"    Step {i+1}/{len(tasks)}: {result[:40]}")
            except Exception as e:
                self.fail(e); raise

        self.complete(f"All {len(tasks)} tasks done")
        return self.state

# Demo: run, "crash", resume
tasks = [f"Document {i}" for i in range(10)]

print("Durable Agent:\n")
agent = DurableAgent(job_id="demo-001")
agent.run(tasks[:5])  # Process 5, then "crash"

print(f"\n  Simulating crash and resume...\n")
agent2 = DurableAgent(job_id="demo-001")  # Reload same job
agent2.run(tasks)  # Resumes from step 5!

print(f"\n  Status: {agent2.state['status']}")
print(f"  Total steps: {agent2.state['step']}")


> **What just happened?** Agent processed 5 documents, saved state to disk. “Crashed.” New agent instance loaded with same job_id — picked up at step 5 and finished the remaining 5. Zero lost work. The checkpoint file contains: current step, all results so far, error history, timestamps. Replace SQLite path with GCS bucket for cloud durability.


### Step 2 · Async Job API — Submit, Poll, Retrieve

Client submits a job, polls for status, retrieves results when done.

**`02_async_api.py`** · _Block 2 of 3_

ASYNC JOB API — Submit + Poll + Retrieve


In [ ]:
# ASYNC JOB API — Submit + Poll + Retrieve
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
import uuid, time, json, asyncio
from pathlib import Path

app = FastAPI(title="Agent Job API")
JOBS_DIR = Path("./jobs")
JOBS_DIR.mkdir(exist_ok=True)

class JobRequest(BaseModel):
    task: str
    items: list[str] = []

def run_agent_job(job_id, task, items):
    """Background worker: runs the agent, saves progress."""
    state = {"id":job_id, "status":"running", "progress":0, "total":len(items), "results":[]}
    path = JOBS_DIR / f"{job_id}.json"
    for i, item in enumerate(items):
        time.sleep(1)  # Simulate LLM call
        state["results"].append(f"Processed: {item}")
        state["progress"] = i + 1
        with open(path, "w") as f: json.dump(state, f)
    state["status"] = "completed"
    with open(path, "w") as f: json.dump(state, f)

@app.post("/jobs")
async def create_job(req: JobRequest, bg: BackgroundTasks):
    job_id = str(uuid.uuid4())[:8]
    state = {"id":job_id, "status":"queued", "progress":0, "total":len(req.items)}
    with open(JOBS_DIR/f"{job_id}.json","w") as f: json.dump(state,f)
    bg.add_task(run_agent_job, job_id, req.task, req.items)
    return {"job_id": job_id, "status": "queued"}

@app.get("/jobs/{job_id}")
async def get_status(job_id: str):
    path = JOBS_DIR / f"{job_id}.json"
    if not path.exists(): return {"error":"Job not found"}
    with open(path) as f: return json.load(f)

print("Async Job API:\n")
print("  POST /jobs       → {job_id, status: 'queued'}")
print("  GET  /jobs/{id}  → {status, progress, total, results}")
print("  Client polls until status='completed'")
print("  Background worker saves progress after each step")


### Step 3 · Pub/Sub Triggers — Event-Driven Agent Execution

Trigger agents from events: new file uploaded, form submitted, scheduled time.

**`03_pubsub_agent.py`** · _Block 3 of 3_

PUB/SUB TRIGGERED AGENT — Event-driven execution — Agent runs when a message arrives on a Pub/Sub topic


In [ ]:
# PUB/SUB TRIGGERED AGENT — Event-driven execution
# Agent runs when a message arrives on a Pub/Sub topic
from fastapi import FastAPI, Request
import base64, json

app = FastAPI()

@app.post("/")
async def pubsub_handler(request: Request):
    """Cloud Run receives Pub/Sub push messages here."""
    body = await request.json()
    msg_data = body.get("message", {}).get("data", "")
    payload = json.loads(base64.b64decode(msg_data).decode())

    event_type = payload.get("type")
    print(f"Event: {event_type}")

    if event_type == "document_uploaded":
        # Trigger document processing agent
        doc_url = payload["url"]
        print(f"  Processing document: {doc_url}")
        # agent.process_document(doc_url)

    elif event_type == "daily_report":
        # Trigger daily research agent
        topic = payload["topic"]
        print(f"  Generating daily report for: {topic}")
        # agent.deep_research(topic)

    elif event_type == "batch_process":
        # Trigger batch processing agent
        items = payload["items"]
        print(f"  Batch processing {len(items)} items")
        # agent.batch(items)

    return {"status": "processed"}

print("Pub/Sub Agent Triggers:\n")
print("  1. Create topic: gcloud pubsub topics create agent-triggers")
print("  2. Create push subscription pointing to Cloud Run URL")
print("  3. Publish event: gcloud pubsub topics publish agent-triggers \\")
print("       --message '{\"type\":\"document_uploaded\",\"url\":\"gs://...\"}'")
print("  4. Cloud Run receives, agent processes")
print("\n  Use cases: file upload → process, cron → daily report, webhook → respond")


> **What just happened?** Cloud Tasks provides managed queuing with built-in retry, deduplication, and rate limiting. For simple async patterns without complex state, this is often sufficient before reaching for Temporal/Inngest.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
